# docx-extract OCR server (DeepSeek-OCR-2) tren Google Colab (T4)

Notebook nay chay server FastAPI OCR cho MathType OLE (WMF/EMF) -> LaTeX,
dung boi `src/docx_extract/mathocr.py` qua bien moi truong `DOCX_EXTRACT_OCR_ENDPOINT`.

**Kien truc lan nay (on dinh hon ban dau):** dung ngrok binary that (khong qua thu vien
`pyngrok`) + **domain co dinh** (dang ky free tren ngrok dashboard), chay server va tunnel
o che do nen trong 1 cell -> mien runtime Colab con song, URL khong doi giua cac lan chay.

**Truoc khi chay:**
1. Dam bao `Runtime > Change runtime type > T4 GPU` da duoc chon.
2. Copy thu muc `colab_ocr_server/` (chua `ocr_server.py`, `ocr_logic.py`) vao Google Drive,
   vi du tai `My Drive/docx-extract/colab_ocr_server/`.
3. Lay ngrok authtoken tai https://dashboard.ngrok.com/get-started/your-authtoken
4. Dang ky 1 domain tinh mien phi tai https://dashboard.ngrok.com/domains -> "Create Domain"
   (free plan cho 1 domain co dinh, dang `xxxx.ngrok-free.app` hoac tuong tu).
5. (Khuyen nghi) Luu 2 gia tri tren vao Colab Secrets (icon chia khoa o sidebar trai):
   `NGROK_AUTHTOKEN` va `NGROK_DOMAIN` -> khong can go lai token/domain moi lan chay,
   va khong lo lo token khi chia se notebook.

## Cell 1: Mount Google Drive va tro vao thu muc code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Sua duong dan nay neu ban de thu muc o vi tri khac trong Drive
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/docx-extract/colab_ocr_server"

%cd "$DRIVE_PROJECT_DIR"
!ls -la

## Cell 2: Cai dat dependencies + tai ngrok binary (chay 1 lan / phien Colab)

In [ ]:
!apt-get -qq update && apt-get -qq install -y libreoffice-core libreoffice-impress
# Khong pin torch/torchvision: Colab da co san ban tuong thich voi GPU driver/CUDA cua may.
# Pin them torch se gay xung dot voi torchvision co san (vd: torchvision requires torch==2.11.0).
!pip install -q fastapi "uvicorn[standard]" python-multipart \
    transformers==4.46.3 tokenizers==0.20.3 einops addict easydict

# Neu thay dong canh bao "gradio ... requires huggingface-hub<2.0 ... but you have
# huggingface-hub 0.36.2" -> bo qua duoc. gradio co san tren Colab nhung ocr_server.py
# khong dung gradio, nen xung dot nay khong anh huong toi server.

# Tai ngrok binary that (khong dung thu vien python pyngrok nua)
!wget -q -O /content/ngrok-v3-stable-linux-amd64.tgz \
    https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz
!tar xzf /content/ngrok-v3-stable-linux-amd64.tgz -C /content
!/content/ngrok version

## Cell 3: Kiem tra GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

## Cell 4: Cau hinh ngrok (authtoken + domain co dinh)

Uu tien lay tu Colab Secrets (`userdata`). Neu chua luu Secrets, dien truc tiep bien
`NGROK_AUTHTOKEN_FALLBACK` / `NGROK_DOMAIN_FALLBACK` ben duoi.

In [ ]:
from google.colab import userdata

NGROK_AUTHTOKEN_FALLBACK = "DAN_NGROK_AUTHTOKEN_VAO_DAY_NEU_KHONG_DUNG_SECRETS"
NGROK_DOMAIN_FALLBACK = "ten-domain-co-dinh-cua-ban.ngrok-free.app"

try:
    NGROK_AUTHTOKEN = userdata.get("NGROK_AUTHTOKEN")
except Exception:
    NGROK_AUTHTOKEN = NGROK_AUTHTOKEN_FALLBACK

try:
    NGROK_DOMAIN = userdata.get("NGROK_DOMAIN")
except Exception:
    NGROK_DOMAIN = NGROK_DOMAIN_FALLBACK

import os
os.environ["NGROK_DOMAIN"] = NGROK_DOMAIN

!/content/ngrok authtoken "$NGROK_AUTHTOKEN"
print("OCR endpoint se la:", f"https://{NGROK_DOMAIN}/ocr")

## Cell 5: Chay server + ngrok tunnel o che do nen (terminal khong tat)

Ca 2 tien trinh (`uvicorn` va `ngrok`) chay nen bang `&` trong cung 1 shell cell -> mien
runtime Colab (container) con song, chung van chay ngam, khong can giu cell nay o trang
thai "dang chay" mai. Log duoc ghi ra file de kiem tra khi can.

In [ ]:
!nohup uvicorn ocr_server:app --host 0.0.0.0 --port 8000 --log-level info \
    > /content/uvicorn.log 2>&1 &
!nohup /content/ngrok http 8000 --domain="$NGROK_DOMAIN" --log stdout \
    > /content/ngrok.log 2>&1 &
!sleep 10 && echo "--- uvicorn.log ---" && tail -n 20 /content/uvicorn.log
!echo "--- ngrok.log ---" && tail -n 20 /content/ngrok.log

## Cell 6: Kiem tra nhanh /health qua domain co dinh

In [ ]:
import requests

public_url = f"https://{NGROK_DOMAIN}"
r = requests.get(f"{public_url}/health", timeout=10)
print(r.status_code, r.text)

## Cell 7: Test OCR thu voi 1 anh cong thuc

Upload 1 file anh cong thuc mau (PNG/WMF/EMF) len Colab (keo-tha vao panel Files ben trai,
hoac de trong Drive roi sua duong dan `SAMPLE_FILE` duoi day).

In [ ]:
SAMPLE_FILE = "sample_formula.png"  # sua thanh duong dan file anh cong thuc that cua ban

with open(SAMPLE_FILE, "rb") as f:
    resp = requests.post(f"{public_url}/ocr", files={"file": (SAMPLE_FILE, f.read())}, timeout=60)
print(resp.status_code, resp.json())

## Cell 8: Benchmark thong luong that (do so giay/cong thuc)

Chay lap lai N lan voi anh mau de co so lieu that cho workload cua ban, thay vi uoc tinh ly thuyet.

In [ ]:
import time

N = 20
with open(SAMPLE_FILE, "rb") as f:
    payload = f.read()

t0 = time.time()
for _ in range(N):
    requests.post(f"{public_url}/ocr", files={"file": (SAMPLE_FILE, payload)}, timeout=60)
elapsed = time.time() - t0

print(f"{elapsed / N:.2f} giay/cong thuc  ->  ~{N / elapsed * 60:.0f} cong thuc/phut")

## Cell 9: Lenh dat bien moi truong ben may chay docx-extract

Vi dung domain co dinh, ban chi can chay dong nay **1 lan duy nhat** (khong doi moi phien
nhu ngrok random domain truoc day) -> co the them thang vao profile PowerShell cua ban.

In [ ]:
print(f'$env:DOCX_EXTRACT_OCR_ENDPOINT = "{public_url}/ocr"')

---
### Ghi chu
- Phien Colab free van bi ngat sau ~90 phut khong tuong tac, toi da ~12 gio. Domain co dinh
  chi giai quyet van de "URL doi moi lan chay" -- KHONG giai quyet duoc viec Colab tu ngat
  runtime. Van can mo lai notebook va chay lai Cell 5 sau khi runtime bi ngat.
- Neu server tra loi cham/timeout, kiem tra `/content/uvicorn.log` va `/content/ngrok.log`
  (vi du: `!tail -n 50 /content/uvicorn.log`) de xem model co dang tai lan dau khong (lan
  goi OCR dau tien se cham hon do lazy-load model).
- Xem `README.md` trong cung thu muc de biet chi tiet tinh toan VRAM/thong luong.